# Fine-tune RF-DETR Nano dari dataset aplikasi Deteksi Objek

Alur: **ZIP ekspor aplikasi → latih RF-DETR Nano (Apache 2.0) → `.tflite` → masukkan ke aplikasi**.

1. Di aplikasi: Galeri dataset → tombol ekspor. Ambil ZIP dari `Download/DeteksiObjek/`.
2. Jalankan notebook ini di Colab dengan **GPU** (Runtime → Change runtime type → T4 atau lebih baik).
3. Ekspor TFLite RF-DETR butuh **Python 3.12** (dicek di sel pertama).

Perkiraan data: mulai dari 200–500 foto per kelas. Dengan puluhan foto, hasilnya hanya cocok untuk uji coba.

In [ ]:
import sys, subprocess
print(sys.version)
assert sys.version_info[:2] == (3, 12), "Ekspor TFLite RF-DETR hanya terpasang di Python 3.12"
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "GPU tidak terdeteksi")

In [ ]:
%pip install -q "rfdetr[train,tflite]"
!wget -q -O prepare_dataset.py https://raw.githubusercontent.com/zackedus/app/main/training/prepare_dataset.py

## 1. Unggah ZIP dan siapkan dataset (train/valid/test)

In [ ]:
from google.colab import files

uploaded = files.upload()  # pilih dataset_XXXX.zip dari aplikasi
zip_name = next(iter(uploaded))
!python prepare_dataset.py "{zip_name}" rfdetr_dataset

## 2. Latih

- `epochs`: 100 untuk dataset < 500 foto (saran dokumentasi RF-DETR: 100–200 epoch untuk dataset kecil).
- Jika GPU kehabisan memori, turunkan `batch_size` dan naikkan `grad_accum_steps` (hasil kali keduanya tetap ±16).

In [ ]:
from rfdetr import RFDETRNano

model = RFDETRNano()
model.train(
    dataset_dir="rfdetr_dataset",
    epochs=100,
    batch_size=8,
    grad_accum_steps=2,
    lr=1e-4,
    output_dir="output",
)

## 3. Ekspor ke TFLite dan uji di satu foto

In [ ]:
from pathlib import Path
from rfdetr import RFDETRNano

trained = RFDETRNano(pretrain_weights="output/checkpoint_best_total.pth")
trained.export(format="tflite", output_dir="export")
print(sorted(p.name for p in Path("export").iterdir()))

In [ ]:
from pathlib import Path
from rfdetr import RFDETRInference

test_image = next(Path("rfdetr_dataset/test").glob("*.jpg"))
tflite = RFDETRInference("export/inference_model_fp16.tflite")
print(test_image.name, tflite.predict(str(test_image), threshold=0.35))

## 4. Unduh untuk aplikasi

Hasilnya dua file. Taruh keduanya di folder `native/` repo, lalu build ulang (`bash setup.sh`, atau push supaya GitHub Actions membuat APK):

- `custom_model.tflite`
- `custom_model.tflite.rfdetr.json`

Di aplikasi, baris status akan menampilkan **Model: RF-DETR kustom**.

In [ ]:
import shutil
from google.colab import files

shutil.copy("export/inference_model_fp16.tflite", "custom_model.tflite")
shutil.copy("export/inference_model_fp16.tflite.rfdetr.json", "custom_model.tflite.rfdetr.json")
files.download("custom_model.tflite")
files.download("custom_model.tflite.rfdetr.json")